# Module 1 — Build a Local Agent

Welcome to the first rung of the ladder. In this module you'll build a local agent
with the [Claude Agent SDK](https://platform.claude.com/docs/en/agent-sdk/overview) integrated with
**Amazon Bedrock** to consume claude model.

### What you'll do

| Part | What you build |
|------|----------------|
| **1A — The one-liner** | A minimal agent with `query()`. See the agent loop run. It's **stateless**. |
| **1B — The real agent** | Upgrade to `ClaudeSDKClient` and add the things that make an agent *good*: a **system prompt**, **`CLAUDE.md`** memory, a custom **skill**, **multi-turn** — then tour the full feature set. |
| **All together** | Run the packaged `agent.py` entrypoint end-to-end — the same one Module 2 will deploy. |

> **Credit:** the Chief of Staff agent is adapted from Anthropic's [claude-cookbooks `chief_of_staff_agent` example](https://github.com/anthropics/claude-cookbooks/tree/main/claude_agent_sdk/chief_of_staff_agent). We reuse that public example and adapt it to run on Amazon Bedrock across this workshop's modules.

## The scenario: a Chief of Staff for *TechStart Inc*

**What does a Chief of Staff do?**
A Chief of Staff is the operator who sits beside a startup CEO and owns the cross-functional questions
that don't belong to any single team — *"What's our runway?"*, *"Can we afford to hire 10 engineers?"*,
*"Draft a board update on our Q2 financials."* They pull numbers from finance, talent, and strategy,
synthesize them, and turn them into a clear, defensible recommendation.

**What does our agent do?**
We're building that operator as an **AI agent** for a fictional Series-A startup, *TechStart Inc*
(50 people, ~$500K/month burn, 20 months of runway). You ask a question in plain English, and the agent:

1. **reads the company's real numbers** from `financial_data/`,
2. **runs deterministic Python scripts** for the math (runway, hiring impact, revenue forecast) — no hand-waving,
3. **delegates deep analysis** to specialist subagents (a financial analyst, a recruiter),
4. **returns a board-ready answer**, and writes a report to disk when asked.

**What problem does this solve?**
It's a realistic stand-in for any **knowledge-work assistant** that must combine *private data* +
*domain procedure* + *tools* to produce answers you can actually trust — instead of a chatbot that
guesses. Everything you learn here transfers to a support agent, an analytics agent, an ops agent, etc.

### Architecture

![Chief of Staff agent architecture](images/architecture.png)

*(Mermaid source for this diagram is in `chief_of_staff_agent/flow_diagram.md`.)*

### Where each capability lives — the agent runs from disk

The Claude Agent SDK doesn't keep the agent's knowledge in code — it reads it from the **filesystem**.
In this notebook we only write the *driver* code; the agent's "body" is the files in
`chief_of_staff_agent/`. That's why every run below points at `cwd="chief_of_staff_agent"`.

| Capability | Where it lives | SDK mechanism |
|------------|----------------|---------------|
| Always-on company facts | `CLAUDE.md` | loaded via `setting_sources=["project"]` |
| Domain procedure ("how we analyze") | `.claude/skills/financial-analysis/` | the `Skill` tool |
| Specialist delegation | `.claude/agents/` | the `Task` tool |
| Deterministic math + data | `scripts/` + `financial_data/` | `Bash` + `Read` |
| Reusable parameterized prompts | `.claude/commands/` | slash commands |
| Communication voice | `.claude/output-styles/` | `settings` |
| Automatic audit logging | `.claude/hooks/` → `audit/` | PostToolUse hooks |

We'll switch each of these on, one at a time, so you see exactly what it adds.

## Setup

Run the cell below to install all dependencies and register the Jupyter kernel.
After it completes, **select the `module-1-local-agent` kernel** from the kernel picker (top-right)
and continue with the rest of the notebook.

## Setup step 1

When you run the first script, it will ask you to select a environment

![](images/select-system-python.png)

and you can select the global env for now, and in this case it is 3.11.15 but this version may change

![](images/select-python-global-env.png)

once selected, you can rerun the setup.sh script

In [ ]:
!bash setup.sh

### Setup step 2

once you see the depencies and kernel spec module-xxx are installed per message from the last step, please refresh your brower (not refresh kernel but brower)

![](images/refresh-browser.png)

and once refresh, click on the button (it probably shows a python verion 3.11.15) you used to select kernel in the preview section

![](images/current-python.png)

it will show you the option to select another kernel and please click

![](images/select-another-kernel.png)

once click, you will see the option of select jupyter kernel and please click on "Jupyter Kernel"

![](images/select-jupyter-kernel.png)

once click, you can see our registered module-x kernel, and the example shows modul-1-x but ***please select accordingly depedning on which model you are working on, if it is other module 2, then select module-2-xx for example***

![](images/example-select-module-1-jupter-kernel.png)

once selected, you will see the following as your kernel , ***please select accordingly depedning on which model you are working on, if it is other module 2, select module-2-xx kernel***

![](images/example-module-1-jupyter-kernel-selected.png)


In [ ]:
# check permissions
import os
from dotenv import load_dotenv

# 1) Load .env (model IDs, region) from this module folder
load_dotenv()

# 2) Use Amazon Bedrock as the model provider.
#    With this set, the SDK reads ANTHROPIC_MODEL from the environment — so we never
#    hardcode a model in code, and you can switch models from .env alone.
os.environ["CLAUDE_CODE_USE_BEDROCK"] = "1"

# 3) Rendering helpers (copied from the Claude cookbook) for pretty notebook output
from utils.agent_visualizer import (
    print_activity,          # stream a one-line summary of each agent step
    display_agent_response,  # render the final answer as an HTML card
    visualize_conversation,  # render the full turn-by-turn conversation
    reset_activity_context,  # reset subagent tracking between runs
)

print("✅ Provider: Amazon Bedrock")
print(f"   Model:            {os.getenv('ANTHROPIC_MODEL', 'NOT SET')}")
print(f"   Small/fast model: {os.getenv('ANTHROPIC_SMALL_FAST_MODEL', 'NOT SET')}")
print(f"   AWS region:       {os.getenv('AWS_REGION', 'NOT SET')}")

In [ ]:
# Optional: confirm AWS credentials + Bedrock model access before we start.
# Safe to skip — but it catches the most common setup problem early.
import boto3

try:
    ident = boto3.client("sts").get_caller_identity()
    print(f"✅ AWS identity: {ident['Arn']}")

    region = os.getenv("AWS_REGION", "us-west-2")
    bedrock = boto3.client("bedrock", region_name=region)
    models = bedrock.list_foundation_models(byProvider="Anthropic")["modelSummaries"]
    print(f"✅ Bedrock reachable in {region}: {len(models)} Anthropic models available")
except Exception as e:
    print(f"⚠️  Could not verify AWS/Bedrock access: {e}")
    print("   Make sure `aws sts get-caller-identity` works and Bedrock model access is enabled.")

---
# Part 1A — The one-liner (`query()`)

The fastest way to see an agent work is the `query()` function. You hand it a **prompt** and a list of
**allowed tools**, and it runs the agent loop — *gather context → take action → verify* — on its own
until it has an answer.

We point it at `chief_of_staff_agent/` and allow just `Read` and `Bash`, so it can read the company's
data files and run calculations. Notice how little code this takes.

### The agent loop

Every capable agent runs the same fundamental loop — it mirrors how a developer actually works:

![The agent loop: gather context, take action, verify](images/claude_agent_sdk_loop.png)

1. **Gather context** — read files, search, inspect data, recall earlier steps (just-in-time, not everything up front).
2. **Take action** — use **tools** to change the world or fetch information: run a command, write a file, query a database, or call an external service via the **Model Context Protocol (MCP)**.
3. **Verify work** — did the test pass, did the query return rows, does the output match the rules? If not, loop back and try again.

This loop — *gather → act → verify → repeat* — is the heartbeat of an agent. The `query()` call below runs exactly this loop on its own until it has an answer.

In [ ]:
from claude_agent_sdk import ClaudeAgentOptions, query

messages = []
async for msg in query(
    prompt="What is our current monthly burn rate? Read the financial data to find out.",
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Bash"],
        cwd="chief_of_staff_agent",   # the agent's files live here
    ),
):
    print_activity(msg)
    messages.append(msg)

display_agent_response(messages)

### `query()` is stateless

Each `query()` call is independent — it remembers **nothing** from the last one. Ask a follow-up that
depends on the previous answer and you'll see the agent has no idea what "that" refers to.

In [ ]:
messages = []
async for msg in query(
    prompt="Based on that, how many months of runway do we have?",  # "that" — but query() forgot
    options=ClaudeAgentOptions(
        allowed_tools=["Read", "Bash"],
        cwd="chief_of_staff_agent",
    ),
):
    print_activity(msg)
    messages.append(msg)

display_agent_response(messages)

The agent had to re-discover the burn rate from scratch — it never saw the first exchange. That's
perfect for a single, self-contained task, and exactly the limitation **Part 1B** removes.

---
# Part 1B — The real agent (`ClaudeSDKClient` + context)

In Part 1A, `query()` answered each question and then **forgot everything**. That statelessness is the
headline difference between the two ways of running an agent — but it isn't the only one. Before we
upgrade, here's the full picture:

### `query()` vs. `ClaudeSDKClient`

| Feature | `query()` | `ClaudeSDKClient` |
|---------|-----------|-------------------|
| **Session** | Creates a new session by default | Reuses the same session |
| **Conversation** | Single exchange | Multiple exchanges in the same context |
| **Connection** | Managed automatically | Manual control (you open & close it) |
| **Streaming input** | ✅ Supported | ✅ Supported |
| **Interrupts** | ❌ Not supported | ✅ Supported |
| **Hooks** | ✅ Supported | ✅ Supported |
| **Custom tools** | ✅ Supported | ✅ Supported |
| **Continue chat** | Manual via `continue_conversation` / `resume` | ✅ Automatic (same open client) |
| **Use case** | One-off tasks | Continuous conversations |

**Rule of thumb:** use `query()` for a single, self-contained task where state doesn't matter; use
`ClaudeSDKClient` when you need a real conversation, fine-grained control, or interrupts. Both run the
*same* agent loop and support the *same* tools, hooks, and context — what differs is the **session
lifecycle**.

Real applications need that conversation. So we now upgrade to **`ClaudeSDKClient`**, which keeps a
session open, and we add the **context** that turns a working agent into a genuinely good one — layering
it in one piece at a time so you can see what each addition buys.

### Under the hood: what the SDK gives you

The [**Claude Agent SDK**](https://platform.claude.com/docs/en/agent-sdk/overview) gives you the *same* agent loop, tools, and context management that power **Claude Code** — so you can build an agent for *anything* (research, analysis, ops), not just coding, on the same proven runtime.

![Claude Agent SDK architecture](images/claude_agent_sdk_architecture.png)

The core idea is to **give the agent a computer**: file access (`Read`/`Write`/`Edit`/`Glob`/`Grep`), a shell (`Bash`), external services via **MCP**, and on-demand **skills**. Everything you switch on below is one of these primitives.

## 1 — A system prompt: shape behavior

A **system prompt** sets the agent's role, standards, and tone. It's how you make the agent behave
**consistently** instead of relying on whatever it does by default. Here we tell it that it *is* the
Chief of Staff and what resources it has.

In [ ]:
from claude_agent_sdk import ClaudeSDKClient

SYSTEM_PROMPT = """You are the Chief of Staff for TechStart Inc, a 50-person startup.

You have company data in the financial_data/ directory, and custom Python scripts in scripts/
that you can run with Bash:
  - python scripts/financial_forecast.py : financial modeling
  - python scripts/hiring_impact.py      : hiring cost / impact
  - python scripts/simple_calculation.py : runway & burn math

Be concise and decision-oriented — you are briefing a busy CEO."""

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Read", "Bash"],
        cwd="chief_of_staff_agent",
    )
) as agent:
    await agent.query("Who are you, and what can you help me with? One short paragraph.")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

## 2 — `CLAUDE.md`: always-on memory

`CLAUDE.md` is a file of project context the agent **always** has loaded — company facts, conventions,
and rules. To load it (and the rest of the `.claude/` settings), we set
**`setting_sources=["project"]`**. Without that, the SDK runs in isolation and ignores these files.

Watch: we ask about runway **without telling it any numbers**. It answers from `CLAUDE.md`
(~$500K monthly burn, 20 months runway, $10M in the bank).

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Read", "Bash"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],   # ← loads CLAUDE.md, skills, subagents, commands
    )
) as agent:
    await agent.query("What's our current runway and cash position?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

## 3 — Skills: domain expertise on demand

`CLAUDE.md` carries *static facts*. A **skill** carries *procedure* — "how we do it here." We've added a
skill at `.claude/skills/financial-analysis/SKILL.md` that defines our standard operating procedure for a
board-ready financial analysis: read the data → run the right script → present headline → drivers →
recommendation.

Skills load via **progressive disclosure**: the agent always sees the skill's *name + description*, and
only pulls in the full instructions when a relevant question comes up. We enable them by allowing the
**`Skill`** tool. Ask a financial question and watch the `financial-analysis` skill get invoked.

> **Skill vs. CLAUDE.md vs. subagent** — three different tools:
> - **`CLAUDE.md`** = always-on facts (the company numbers)
> - **Skill** = a reusable procedure ("how we run this analysis")
> - **Subagent** = delegating a whole task to a specialist (we'll see this below)

#### What *is* an agent skill?

Agent Skills are a lightweight, **open format** for extending an agent's capabilities. A skill is a folder of organized files — instructions, scripts, assets — that the agent discovers and loads to perform a task accurately. It's an open standard used across Claude Code, Codex, Gemini CLI, and more, so a skill you write in one environment works in others.

![An agent skill is a folder: a SKILL.md plus scripts and resources](images/agent-skills-structure.png)

*Figure: the folder structure of an Agent Skill — instructions, scripts, and resources organized for the agent to use domain-specific expertise.* *Image from the [Agent Skills with Anthropic short course](https://www.deeplearning.ai/short-courses/agent-skills-with-anthropic/).*

**Why skills?** A bare agent with bash and a filesystem is easy to reason about — but it lacks the *domain expertise* to do **your** work **your** way. Skills supply that, on demand:

1. **Domain expertise** — your methodologies and conventions, not just generic behavior.
2. **Repeatable workflows** — articulated steps that give consistent results across runs and users.
3. **New capabilities** — things the model can't do out of the box (generate a report, run a custom script).

**How they stay cheap — progressive disclosure.** You might have hundreds of skills, so they load **only what's needed, when it's needed**:

| Layer | When it loads |
|-------|---------------|
| **Metadata** (name + description) | Always — so the agent knows the skill exists |
| **Instructions** (`SKILL.md` body) | When the skill is triggered |
| **Resources** (reference files, scripts) | Only as needed |

Scripts run **outside** the context window, so the agent pulls in only the information it actually needs — the heart of **context engineering**.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Skill", "Read", "Bash"],   # ← Skill tool enabled
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
    )
) as agent:
    await agent.query(
        "Give me a board-ready analysis of our runway if we hire 10 engineers."
    )
    async for msg in agent.receive_response():
        print_activity(msg)   # look for: 🤖 Using: Skill(financial-analysis)
        messages.append(msg)

display_agent_response(messages)

## 4 — Multi-turn: the agent remembers (within the session)

Because `ClaudeSDKClient` keeps the session open, we can ask **follow-ups** and the agent remembers the
earlier turns of *this* conversation — the exact thing `query()` couldn't do in Part 1A. We send two
queries to the **same open client** and the second relies on the first.

(This is *in-session* memory only — when the process ends, it's gone. Persisting memory across sessions
and deployments is what **Module 3** adds.)

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Skill", "Read", "Bash"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
    )
) as agent:
    # Turn 1
    await agent.query("What is our current monthly burn rate?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

    # Turn 2 — "that" refers back to turn 1; the open session remembers
    await agent.query("If that increased by 20%, how many months of runway would we have left?")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

visualize_conversation(messages)

---
## The full feature set

The Claude Agent SDK gives you more than a system prompt and a skill. Below we tour the rest of what's
already wired into `chief_of_staff_agent/`. Each builds on the same `ClaudeSDKClient` pattern — only the
options change.

### Custom scripts via the `Bash` tool

The agent isn't limited to the model's head-math — it runs **real Python scripts** in `scripts/` through
the `Bash` tool. This keeps calculations deterministic and auditable.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Read", "Bash"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
    )
) as agent:
    await agent.query(
        "Use the simple_calculation script with total cash 10000000 and monthly burn 500000."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

### Output styles — same answer, different voice

**Output styles** (in `.claude/output-styles/`) change *how* the agent communicates without changing
*what* it does. We ask the identical question twice — once `executive`, once `technical` — by passing
`settings='{"outputStyle": "..."}'`. (Output styles are filesystem settings, so `setting_sources` must
include `"project"`.)

In [ ]:
question = "Summarize our financial position in two sentences."

# Executive voice
messages_exec = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT, cwd="chief_of_staff_agent",
        setting_sources=["project"], settings='{"outputStyle": "executive"}',
    )
) as agent:
    await agent.query(question)
    async for msg in agent.receive_response():
        messages_exec.append(msg)

# Technical voice
messages_tech = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT, cwd="chief_of_staff_agent",
        setting_sources=["project"], settings='{"outputStyle": "technical"}',
    )
) as agent:
    await agent.query(question)
    async for msg in agent.receive_response():
        messages_tech.append(msg)

print("───────── EXECUTIVE ─────────")
display_agent_response(messages_exec)
print("───────── TECHNICAL ─────────")
display_agent_response(messages_tech)

### Plan mode — think before acting

With `permission_mode="plan"`, the agent **plans without executing** — no files written, no commands
run. It's ideal for reviewing the agent's intended approach to a high-stakes decision before letting it
act.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Read", "Bash", "Write"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
        permission_mode="plan",   # ← think only, don't execute
    )
) as agent:
    await agent.query(
        "I'm considering acquiring competitor SmartDev for $8M. How would you approach analyzing this?"
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

### Custom slash commands

Slash commands (in `.claude/commands/`) are reusable, parameterized prompts. `/budget-impact <decision>`
expands into a full instruction that, here, delegates to the financial-analyst subagent. The user types
a few words; the command supplies the rigor.

In [ ]:
messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Task", "Read", "Bash"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
    )
) as agent:
    await agent.query("/budget-impact hiring 5 senior backend engineers in Q3")
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

### Hooks — automatic, deterministic actions

**Hooks** (wired in `.claude/settings.local.json`) run code on tool events — no prompting needed. This
agent has a `PostToolUse` hook on `Bash` that logs every script the agent runs to
`audit/script_usage_log.json`. After the run above, the log has grown automatically.

In [ ]:
import json
from pathlib import Path

log_path = Path("chief_of_staff_agent/audit/script_usage_log.json")
if log_path.exists():
    data = json.loads(log_path.read_text())
    entries = data.get("script_executions", [])   # the file is a dict, not a bare list
    print(f"📋 audit/script_usage_log.json contains {len(entries)} script execution(s)")
    if entries:
        print("\nMost recent entry:")
        print(json.dumps(entries[-1], indent=2))
else:
    print("No audit log yet — run a cell that uses the Bash tool first.")

### Skills vs. Tools vs. MCP vs. Subagents

These four often get confused. A quick map:

| | Skills | Tools | MCP | Subagents |
|---|--------|-------|-----|-----------|
| **Provides** | Procedural knowledge | Low-level capabilities | Tool connectivity to external systems | Task delegation |
| **Loads** | Dynamically, as needed | Always in context | Always connected | When invoked |
| **Best for** | "How *we* do it" | Reading, running, querying | Bringing in external data/tools | Parallel, isolated work |

![Skills vs. tools, MCP, and subagents](images/skills-vs-others.png)

*Image from the [Agent Skills with Anthropic short course](https://www.deeplearning.ai/short-courses/agent-skills-with-anthropic/).*

A useful way to hold it: **tools** are low-level capabilities (a hammer and nails); **skills** are higher-level knowledge (how to build a bookshelf). **MCP** brings in external data and tools the model doesn't ship with; **subagents** are separate agent instances for parallel, isolated work, each with their own tools and context. When to reach for what:

- **Tools** — low-level capabilities that should always be available.
- **Skills** — procedural, repeatable workflows that load only when relevant.
- **MCP** — connecting to external data and systems (databases, APIs, SaaS).
- **Subagents** — parallelizing or isolating specialized tasks (you'll see one next).

### Subagents — delegate to specialists

The Chief of Staff can hand off whole tasks to **subagents** (in `.claude/agents/`) via the **`Task`**
tool. We ship two — `financial-analyst` and `recruiter` — each with its own focused instructions and
tools. The main agent stays a coordinator; the specialists do the deep work in their own context.

In [ ]:
reset_activity_context()  # clean subagent tracking for a fresh visualization

messages = []
async with ClaudeSDKClient(
    options=ClaudeAgentOptions(
        system_prompt=SYSTEM_PROMPT,
        allowed_tools=["Task", "Read", "Bash", "WebSearch"],
        cwd="chief_of_staff_agent",
        setting_sources=["project"],
    )
) as agent:
    await agent.query(
        "Delegate to the financial-analyst: assess whether we can afford to hire 10 engineers "
        "this year given our runway."
    )
    async for msg in agent.receive_response():
        print_activity(msg)
        messages.append(msg)

display_agent_response(messages)

---
# Putting it all together

Everything above used `ClaudeAgentOptions` written out inline, so you could see each piece. In a real
project you'd **package** that configuration once. We've done exactly that in
`chief_of_staff_agent/agent.py`, which exposes a single `send_query()` helper that wires up the system
prompt, tools, `setting_sources`, and `cwd` — and (because there's no hardcoded `model=`) picks up the
Bedrock model from your environment.

This is the shape of a deployable agent: one entrypoint, all capability behind it. **Module 2 deploys
this exact `agent.py`.**

In [ ]:
from chief_of_staff_agent.agent import send_query

# One call exercises memory (CLAUDE.md) + the skill + scripts together.
result, messages = await send_query(
    "Give me a one-page strategic brief: our runway, the cost of the planned 10 engineering hires, "
    "and your recommendation."
)

display_agent_response(messages)

### Multi-turn through the packaged entrypoint

`send_query()` also supports `continue_conversation=True`, so the packaged agent can hold a back-and-forth
just like the open client did earlier.

In [ ]:
result, messages = await send_query(
    "Of those hires, which roles should we prioritize first?",
    continue_conversation=True,
)
display_agent_response(messages)

---
## Key takeaways

- An agent is a **model in a loop with tools** — `query()` shows how little code that takes, and that it's **stateless**.
- **`ClaudeSDKClient`** is **stateful and multi-turn** — it remembers earlier turns *within a session*.
- **Context engineering** is what makes an agent reliable: a **system prompt** shapes behavior, **`CLAUDE.md`** supplies always-on facts, **skills** supply procedure on demand, and **subagents/commands/hooks/output-styles** extend it further.
- Loading any of the filesystem pieces (CLAUDE.md, skills, subagents, commands, output styles, hooks) requires **`setting_sources=["project"]`**.
- The model is chosen by **`ANTHROPIC_MODEL` on Bedrock** — no hardcoded model in code.
- All of this still runs **on your laptop**. Next, we put it in production.

## Next steps

Continue to **Module 2: Deploy to AgentCore Runtime** to take this exact agent
(`chief_of_staff_agent/agent.py`) to a managed, serverless runtime.